#Iniciando instalando o Unsloth

In [ ]:
!pip install unsloth

#Carregando Modelo base (Fazendo a configuraçao do QLoRA)

In [ ]:
from unsloth import FastLanguageModel


# carregando e tokenizando o modelo em 4 bits
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # versao usando o llama-3-8b
    max_seq_length = 2048,  # Set do tamanho máximo de contexto
    dtype = None,           # o None permito o unsloth detectar o float type
    load_in_4bit = True,    # ativacao do Qlora
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.1k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


In [ ]:

# Configurando a parte de hiperparametros do LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,                # Rank de memoria de matrizes para o lora
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,       # fator de escala do LoRA
    lora_dropout = 0,      # evita um pouco de lentidao
    bias = "none",         # LoRA sem vieses matematico
    use_gradient_checkpointing = "unsloth", # evita estoura a Vram do unsloth
    random_state = 3350,
)

Unsloth 2026.5.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


#Carregando e Explorando o DataSet

In [ ]:
from datasets import load_dataset

# Carregando o DataSet, o split="train" guarda o dataset bruto no dsb
dsb = load_dataset("STEM-AI-mtl/Electrical-engineering", split="train")

# Usando 20% para e treino e seed fixando o embaralhamento
dataset_split = dsb.train_test_split(test_size=0.2, seed=3350)

# Separando a variavel de treino e de teste
dataset_train = dataset_split["train"]
dataset_test = dataset_split["test"]

# Apenas verificando o tamanho
print("----Distribuiçao do dataset----")
print(f"Quantidade Total de Registros: {len(dsb)}")
print(f"Quantidade para o Treino (80%): {len(dataset_train)}")
print(f"Quantidade para o Teste (20%): {len(dataset_test)}")

print("\n\n Apenas mostrando um registro bruto como exemplo:")
print(dataset_train[0])

README.md:   0%|          | 0.00/508 [00:00<?, ?B/s]

Electrical-engineering.json:   0%|          | 0.00/646k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1131 [00:00<?, ? examples/s]

----Distribuiçao do dataset----
Quantidade Total de Registros: 1131
Quantidade para o Treino (80%): 904
Quantidade para o Teste (20%): 227


 Apenas mostrando um registro bruto como exemplo:
{'instruction': 'You are an electrical engineer and you will answer questions related to electrical engineering.', 'input': 'What is the purpose of a flyback diode in a circuit with an inductive load?', 'output': 'A flyback diode in a circuit with an inductive load is used to protect other components from voltage spikes that occur when the current to the inductive load is suddenly switched off. The diode provides a path for the inductive kickback, preventing damage to the circuit.'}


#Definindo a Formataçao do prompt template

In [ ]:
# Utilizando o template oficial se baseando nos tokens de estrutura do Llama-3-Instruct

LLAMA3_TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Responda à questão técnica de forma clara, precisa e profissional:<|eot_id|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{}<|eot_id|>"""


# funcao para processar e juntar as colunas do dataset
def template_llama3(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]

    lista_textos = []


    # junta as colunas em uma estrutura de texto corrido
    for inst, inp, out in zip(instructions, inputs, outputs):

        # verifica o 'input': se houver dados é unido a instruçao
        if inp and str(inp).strip():
            prompt_completo = f"{inst}\n\n[Contexto/Dados Técnicos]:\n\n{inp}"
        else:
            prompt_completo = inst

        # Adiciona os textos estruturados
        texto_final = LLAMA3_TEMPLATE.format(prompt_completo, out)
        lista_textos.append(texto_final)

    return { "text" : lista_textos }

# faz o mapeamento em lote
dataset_train = dataset_train.map(template_llama3, batched = True)
dataset_test  = dataset_test.map(template_llama3, batched = True)

# visualizaçao do resultado formatado
print("---- Resultado formatado ----")
print(dataset_train[0]["text"])

Map:   0%|          | 0/904 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

---- Resultado formatado ----
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Responda à questão técnica de forma clara, precisa e profissional:<|eot_id|><|start_header_id|>user<|end_header_id|>

You are an electrical engineer and you will answer questions related to electrical engineering.

[Contexto/Dados Técnicos]:

What is the purpose of a flyback diode in a circuit with an inductive load?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

A flyback diode in a circuit with an inductive load is used to protect other components from voltage spikes that occur when the current to the inductive load is suddenly switched off. The diode provides a path for the inductive kickback, preventing damage to the circuit.<|eot_id|>


# Interferencia do modelo base antes do treino

In [ ]:

FastLanguageModel.for_inference(model)

# seleciona uma amostra do dataset_test que o modelo nao viu
amostra = dataset_test[111]
persona_padrao = "You are an electrical engineer and you will answer questions related to electrical engineering."
pergunta_teste = amostra["input"] if amostra["input"] and str(amostra["input"]).strip() else amostra["instruction"]


# montando o prompt de test
prompt_teste = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{}<|eot_id|>
<|start_header_id|>user<|end_header_id|>

{}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>

"""

# Tokenização da pergunta e o envio dos tensores para a memoria da GPU (CUDA)
inputs = tokenizer(
    [prompt_teste.format(persona_padrao, pergunta_teste)],
    return_tensors = "pt"
).to("cuda")


# execunto a geraçao de texto pelo modelo base original
outputs = model.generate(
    **inputs,
    max_new_tokens = 256,
    use_cache = True
)


# mostrando os resultados comparativos

print("\n\n---- Questao Selecionada----")
print(pergunta_teste)
print("\n\n---- Resposta esperada: ----")
print(amostra["output"])
print("\n\n---- Resposta do modelo sem fine_tuning ----")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].split("assistant")[-1].strip())

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/



---- Questao Selecionada----
What are the key considerations when designing a PCB layout for high-frequency circuits?


---- Resposta esperada: ----
When designing a PCB layout for high-frequency circuits, key considerations include minimizing trace lengths to reduce signal attenuation, using controlled impedance traces, avoiding sharp angles in trace routing, implementing proper grounding techniques, and carefully placing components to minimize electromagnetic interference.


---- Resposta do modelo sem fine_tuning ----
When designing a PCB layout for high-frequency circuits, there are several key considerations to ensure the integrity and performance of the design. Here are some of the most important ones:

1. **Signal Integrity**: High-frequency circuits are prone to signal integrity issues such as ringing, crosstalk, and reflections. To mitigate these issues, it's essential to ensure that the PCB layout minimizes signal delay, reduces signal reflections, and minimizes coupling be

# Fazendo o Treinamento

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported


# Configuraoes do Treinador e as otimizaçoes do Unsloth
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = dataset_test,         # verifica a evolução dos erros nos dados de teste
    dataset_text_field = "text",         # busca aquela coluna unificada
    max_seq_length = 2048,               # Limita o contexto herdado
    dataset_num_proc = 2,                # acelera o input de dados usando o multi processamento

    packing = False,                     # isola as estruturas  de dialogo para maior precisao


    # definindo os hiperparâmetros para o treino da gpu do colab
    args = TrainingArguments(
        per_device_train_batch_size = 4,  # amostras processadas por vez na memoria da gpu
        gradient_accumulation_steps = 2,  # junta gradientes para simular a estabilidade de um lote de 8 (2x4)
        warmup_steps = 5,                 # inicia o aprendizado aos poucos
        max_steps = 60,                   # limite de interaçoes
        learning_rate = 2e-4,             # taxa de aprendizado (velocidade de ajuste dos pesos LoRA)
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),   # troca para o Bfloat16 se o Colab fornecer uma gpu moderna
        logging_steps = 1,                # status para a metrica de perda (loss)
        optim = "adamw_8bit",             # economiza aproximadamente 75% de ram
        weight_decay = 0.01,              # evita que o modelo apenas memorize
        lr_scheduler_type = "linear",     # decaimento linear para a taxa de aprendizado
        seed = 3350,
        output_dir = "outputs",           #



        report_to = "none",
        disable_tqdm = False,
    ),
)

# startando o treino
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/904 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/227 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 904 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 167,772,160 of 8,198,033,408 (2.05% trained)


Step,Training Loss
1,0.489935
2,0.508660
3,0.520435
4,0.478402
5,0.494440
6,0.535930
7,0.507073
8,0.400336
9,0.287640
10,0.335736


# Verificando depois do treino

In [ ]:
# desativa o backpropagation
FastLanguageModel.for_inference(model)

# compara com a quetsao feita anteriormente
amostra_pos = dataset_test[111]
persona_padrao = "You are an electrical engineer and you will answer questions related to electrical engineering."
pergunta_teste = amostra_pos["input"] if amostra_pos["input"] and str(amostra_pos["input"]).strip() else amostra_pos["instruction"]

# utilizando o mesmo padrao do prompt

prompt_teste = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{}<|eot_id|>
<|start_header_id|>user<|end_header_id|>

{}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>

"""

# tekeniza e envia a pergunta
inputs = tokenizer(
    [prompt_teste.format(persona_padrao, pergunta_teste)],
    return_tensors = "pt"
).to("cuda")

# solicita a resposta ao modelo treinado
outputs = model.generate(
    **inputs,
    max_new_tokens = 256,
    use_cache = True
)

# mostra o resultado de antes e depois
print("---- Mesma questao usada anteriormente ----")
print(pergunta_teste)
print("\n\n ---- Resposta esperada: ----")
print(amostra_pos["output"])
print("\n\n Resposta do modelo treinado: ----")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].split("assistant")[-1].strip())

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


---- Mesma questao usada anteriormente ----
What are the key considerations when designing a PCB layout for high-frequency circuits?


 ---- Resposta esperada: ----
When designing a PCB layout for high-frequency circuits, key considerations include minimizing trace lengths to reduce signal attenuation, using controlled impedance traces, avoiding sharp angles in trace routing, implementing proper grounding techniques, and carefully placing components to minimize electromagnetic interference.


 Resposta do modelo treinado: ----
Key considerations for designing a PCB layout for high-frequency circuits include minimizing trace length and routing for low impedance, using proper grounding techniques to reduce noise, and ensuring that the layout is symmetrical to reduce electromagnetic interference (EMI). Additionally, using high-frequency capacitors and inductors, and carefully placing decoupling capacitors are important.


# Grafico de perdas

In [ ]:
import matplotlib.pyplot as plt

# pega o historico de passos e perdas

'''historico = trainer.state.log_history
passos = [registro["step"] for registro in historico if "loss" in registro]
perda_treino = [registro["loss"] for registro in historico if "loss" in registro]
'''

# grafico

'''plt.figure(figsize=(10, 5))
plt.plot(passos, perda_treino, label="Training Loss", color="#1f77b4", linewidth=2, marker='o', markersize=4)

plt.title("Curva de Aprendizado - Fine-Tuning Llama-3 ", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Passos de Treinamento (Steps)", fontsize=12)
plt.ylabel("Função de Perda (Loss)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()'''

'plt.figure(figsize=(10, 5))\nplt.plot(passos, perda_treino, label="Training Loss", color="#1f77b4", linewidth=2, marker=\'o\', markersize=4)\n\nplt.title("Curva de Aprendizado - Fine-Tuning Llama-3 ", fontsize=14, fontweight=\'bold\', pad=15)\nplt.xlabel("Passos de Treinamento (Steps)", fontsize=12)\nplt.ylabel("Função de Perda (Loss)", fontsize=12)\nplt.grid(True, linestyle="--", alpha=0.6)\nplt.legend(fontsize=11)\n\nplt.tight_layout()\nplt.show()'